# Chapter 7 — Unsupervised Learning

**Mata Kuliah:** Deep Learning
**Buku:** Practical Statistics for Data Scientists
**Chapter:** 7 — Unsupervised Learning

## Tujuan

Notebook ini mereproduksi kode Python dan merangkum konsep
**unsupervised learning**, yaitu metode statistik yang mengekstrak
makna dari data **tanpa melatih model pada data berlabel** (data yang
hasil atau outcome-nya sudah diketahui).

Pembahasan:
1. Principal Components Analysis (PCA)
2. Correspondence Analysis
3. K-Means Clustering
4. Hierarchical Clustering
5. Model-Based Clustering
6. Scaling dan Categorical Variables (termasuk Gower's distance)

Notebook dijalankan di **Google Colab**. Dataset dibaca langsung dari
repository resmi buku.

## Pendahuluan

Istilah *unsupervised learning* merujuk pada metode statistik yang
mengekstrak makna dari data **tanpa melatih model pada data
berlabel**. Pada Chapter 4-6 tujuannya membangun model (regression atau
klasifikasi) untuk memprediksi respons. Pada unsupervised learning
tujuannya adalah membangun model *tanpa* respons, yaitu tanpa
"variabel hasil".

Unsupervised learning dapat dipakai untuk beberapa tujuan:

- Membangun **prediktor** tanpa respons, misalnya **reduksi dimensi**
  untuk memperoleh sedikit variabel yang merangkum banyak variabel.
- Membagi data ke dalam kelompok (**clustering**) untuk menemukan
  struktur yang tidak terlihat.
- Sebagai **eksplorasi** awal data besar dan tidak rapi, seperti data
  sensor, sebelum dilakukan pemodelan.
- Membantu **regression/klasifikasi** pada data yang tidak berlabel
  (kasus label mahal atau sulit didapat).

> **Unsupervised Learning dan Prediksi.** Unsupervised learning juga
> berperan dalam regression dan klasifikasi. Dalam **automatic
> machine learning** (misal pada data sensor satelit tanpa label),
> teknik ini dipakai untuk menemukan pola. Cluster yang ditemukan
> dapat dijadikan fitur tambahan bagi model supervised.

## Import Library

In [ ]:
# prince dipakai untuk Correspondence Analysis (opsional, ada fallback manual)
!pip install -q prince

In [ ]:
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import preprocessing
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform

DATA_URL = ('https://raw.githubusercontent.com/gedeck/'
            'practical-statistics-for-data-scientists/main/data/')

%matplotlib inline
sns.set_theme(style="whitegrid")

## Membaca Dataset

- `sp500_px.csv.gz` — harga saham harian perusahaan S&P 500 (sebagai
  *return* harian), indeks berupa tanggal.
- `loan_data.csv.gz` — data pinjaman LendingClub.
- `housetasks.csv` — tabel kontingensi pembagian tugas rumah tangga.

In [ ]:
sp500_px = pd.read_csv(DATA_URL + 'sp500_px.csv.gz', index_col=0)
loan_data = pd.read_csv(DATA_URL + 'loan_data.csv.gz')
loan_data['outcome'] = loan_data['outcome'].astype('category')
housetasks = pd.read_csv(DATA_URL + 'housetasks.csv', index_col=0)

print('sp500_px  :', sp500_px.shape)
print('loan_data :', loan_data.shape)
print('housetasks:', housetasks.shape)
sp500_px.iloc[:5, :6]

# 1. Principal Components Analysis (PCA)

Variabel sering bervariasi bersama (*covary*) dan sebagian variasi pada
satu variabel sebenarnya duplikasi dari variasi variabel lain (misal
total tagihan restoran dan tip). **PCA** adalah teknik untuk menemukan
cara variabel numerik berkovariasi.

### Key Terms

- **Principal component** — kombinasi linear dari variabel predictor.
- **Loadings** — bobot yang mentransformasi predictor menjadi
  komponen. Sinonim: weights.
- **Screeplot** — plot varians komponen, menunjukkan kepentingan
  relatif komponen, baik sebagai explained variance maupun proporsi
  explained variance.

Ide PCA adalah menggabungkan banyak variabel numerik menjadi himpunan
variabel yang lebih kecil, berupa kombinasi linear berbobot dari
himpunan awal. Himpunan lebih kecil ini (principal components)
"menjelaskan" sebagian besar variabilitas himpunan lengkap, sehingga
dimensi data berkurang. Bobot yang membentuk komponen menunjukkan
kontribusi relatif variabel asli terhadap komponen baru.

PCA pertama kali diusulkan oleh **Karl Pearson**, dalam makalah yang
mungkin menjadi makalah pertama tentang unsupervised learning. PCA
dapat dipandang sebagai versi unsupervised dari linear discriminant
analysis.

## Contoh Sederhana

Untuk dua variabel $X_1$ dan $X_2$ ada dua principal component
$Z_i$ ($i = 1, 2$):

$$
Z_i = w_{i,1}X_1 + w_{i,2}X_2
$$

Bobot $w_{i,1}, w_{i,2}$ disebut **component loadings**. Komponen
pertama $Z_1$ adalah kombinasi linear yang paling menjelaskan total
variasi. Komponen kedua $Z_2$ **ortogonal** terhadap yang pertama dan
menjelaskan sebanyak mungkin variasi yang tersisa. Umumnya komponen
dihitung pada deviasi dari mean, bukan nilai asli.

Berikut PCA pada return saham Chevron (CVX) dan ExxonMobil (XOM).

In [ ]:
oil_px = sp500_px[['XOM', 'CVX']]

pcs = PCA(n_components=2)
pcs.fit(oil_px)

loadings = pd.DataFrame(pcs.components_, columns=oil_px.columns,
                        index=['PC1', 'PC2'])
loadings

Komponen pertama memberi bobot kira-kira sama pada kedua saham, yang
berarti pergerakan bersama kedua saham minyak tersebut (pergerakan
pasar sektor energi). Komponen kedua membedakan pergerakan satu saham
terhadap yang lain. Tanda positif/negatif seluruh bobot komponen dapat
dibalik tanpa mengubah maknanya (garis yang sama).

Berikut plot data dengan sumbu komponen:

In [ ]:
def abline(slope, intercept, ax):
    # Hitung koordinat garis dengan slope dan intercept tertentu
    x_vals = np.array(ax.get_xlim())
    return (x_vals, intercept + slope * x_vals)

ax = oil_px.plot.scatter(x='XOM', y='CVX', alpha=0.3, figsize=(5, 5))
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.plot(*abline(loadings.loc['PC1', 'CVX'] / loadings.loc['PC1', 'XOM'], 0, ax),
        '--', color='C1', label='PC1')
ax.plot(*abline(loadings.loc['PC2', 'CVX'] / loadings.loc['PC2', 'XOM'], 0, ax),
        '--', color='C1', label='PC2')
ax.set_title('Sumbu principal component pada return XOM vs CVX')
plt.show()

Komponen pertama (garis panjang) menangkap korelasi kuat antara kedua
saham, sedangkan komponen kedua tegak lurus dan menangkap variasi
sisanya.

## Computing the Principal Components

Untuk lebih dari dua variabel prosesnya serupa:

1. Komponen pertama adalah kombinasi linear variabel predictor yang
   **memaksimalkan persen total varians yang dijelaskan**.
2. Kombinasi linear itu menjadi predictor "baru" pertama, $Z_1$.
3. PCA mengulang proses dengan bobot lain untuk membuat predictor baru
   kedua, $Z_2$, dengan syarat $Z_1$ dan $Z_2$ **tidak berkorelasi**.
4. Proses berlanjut hingga jumlah komponen sama dengan jumlah variabel.
5. Pilih sebanyak komponen yang diperlukan untuk menjelaskan sebagian
   besar varians.
6. Hasilnya adalah bobot untuk tiap komponen. Langkah akhir mengubah
   data asli menjadi **skor** komponen dengan menerapkan bobot pada
   nilai asli. Skor ini dapat dipakai sebagai himpunan predictor yang
   telah direduksi.

PCA bekerja langsung dari **correlation matrix** atau **covariance
matrix** data dan dihitung dengan cepat tanpa iterasi. PCA hanya
bekerja dengan variabel numerik.

## Interpreting Principal Components

Sifat komponen sering mengungkap struktur data. Dua tampilan standar
adalah **screeplot** (kepentingan relatif komponen; sumbu y adalah
eigenvalue) dan plot **loadings**. Contoh berikut memakai beberapa
saham teratas S&P 500 sejak 2011.

In [ ]:
syms = sorted(['AAPL', 'MSFT', 'CSCO', 'INTC', 'CVX', 'XOM', 'SLB', 'COP',
               'JPM', 'WFC', 'USB', 'AXP', 'WMT', 'TGT', 'HD', 'COST'])
top_sp = sp500_px.loc[sp500_px.index >= '2011-01-01', syms]

sp_pca = PCA()
sp_pca.fit(top_sp)

explained_variance = pd.DataFrame(sp_pca.explained_variance_)
ax = explained_variance.head(10).plot.bar(legend=False, figsize=(5, 4))
ax.set_xlabel('Component')
ax.set_ylabel('Explained variance')
ax.set_title('Screeplot')
plt.show()

print('Proporsi varians PC1-PC5:', sp_pca.explained_variance_ratio_[:5].round(3))
print('Kumulatif PC1-PC5      :', sp_pca.explained_variance_ratio_[:5].cumsum().round(3))

Screeplot menunjukkan bahwa varians komponen pertama paling besar,
diikuti penurunan tajam: komponen pertama dan kedua menjelaskan
sebagian besar variabilitas, sedangkan komponen berikutnya bernilai
kecil. Ini menunjukkan data 16 saham dapat diwakili dengan beberapa
komponen saja.

Selanjutnya **loadings** untuk lima komponen teratas:

In [ ]:
loadings = pd.DataFrame(sp_pca.components_[0:5, :], columns=top_sp.columns)
maxPC = 1.01 * np.max(np.max(np.abs(loadings.loc[0:5, :])))

f, axes = plt.subplots(5, 1, figsize=(5, 8), sharex=True)
for i, ax in enumerate(axes):
    pc_loadings = loadings.loc[i, :]
    colors = ['C0' if l > 0 else 'C1' for l in pc_loadings]
    ax.axhline(color='#888888')
    pc_loadings.plot.bar(ax=ax, color=colors)
    ax.set_ylabel(f'PC{i+1}')
    ax.set_ylim(-maxPC, maxPC)
plt.tight_layout()
plt.show()

Interpretasi pada buku:

- Komponen pertama: bobot semua saham positif dengan besar relatif
  sama, mencerminkan **pergerakan pasar secara umum**.
- Komponen kedua: membedakan saham energi (bobot satu tanda) dari saham
  lain, mencerminkan **pergerakan sektor energi** terhadap pasar.
- Komponen ketiga dan seterusnya: pola yang lebih halus, misalnya
  perbedaan antara saham keuangan dan saham teknologi/ritel.

> **How Many Components to Choose?** Jika tujuannya reduksi dimensi,
> pilih jumlah komponen dengan aturan ad hoc: komponen yang
> menjelaskan "sebagian besar" varians, dilihat dari screeplot atau
> dengan memilih komponen teratas hingga **varians kumulatif melebihi
> ambang** (misal 80%). Loadings juga dapat diperiksa apakah
> komponen punya interpretasi intuitif. **Cross-validation** adalah
> metode yang lebih formal.

In [ ]:
# Berapa komponen untuk menjelaskan >= 80% varians?
cum = np.cumsum(sp_pca.explained_variance_ratio_)
print('Jumlah komponen untuk >= 80%% varians: %d' % (np.argmax(cum >= 0.8) + 1))
print(cum.round(3))

### Key Ideas — PCA

- Principal components adalah kombinasi linear dari variabel
  predictor (hanya data numerik).
- Komponen dihitung untuk meminimalkan korelasi antar komponen,
  sehingga mengurangi redundansi.
- Sejumlah kecil komponen biasanya menjelaskan sebagian besar
  variasi pada variabel.
- Screeplot menunjukkan kontribusi relatif tiap komponen pada
  varians total.
- Loadings menunjukkan bobot variabel asli pada tiap komponen.

# 2. Correspondence Analysis

PCA tidak dapat dipakai untuk data kategorikal. Teknik yang agak
terkait adalah **correspondence analysis**. Tujuannya mengenali
**asosiasi antar kategori**, atau antar fitur kategorikal. Kemiripan
dengan PCA terutama di bawah permukaan: aljabar matriks untuk penskalaan
dimensi. Correspondence analysis dipakai terutama untuk analisis grafis
data kategorikal berdimensi rendah dan tidak dipakai seperti PCA untuk
reduksi dimensi pada big data.

Inputnya dapat dipandang sebagai tabel (kontingensi) dengan baris satu
variabel dan kolom variabel lain, sel berisi jumlah record. Keluarannya
(setelah aljabar matriks) adalah **biplot**: scatterplot dengan sumbu
yang diskalakan, dengan persentase yang menunjukkan besar varians yang
dijelaskan oleh dimensi tersebut. Makna unit sumbu tidak terhubung
secara intuitif dengan data asli. Nilai utamanya adalah memperlihatkan
secara grafis variabel yang saling berasosiasi (berdekatan pada plot).

Contoh klasik: **tugas rumah tangga** (`housetasks`) yang dikerjakan
istri, suami, bergantian, atau bersama-sama.

In [ ]:
housetasks

Buku memakai package `prince` (`prince.CA`). Package ini berubah
API-nya antar versi, sehingga di bawah disediakan **implementasi manual**
correspondence analysis (berbasis SVD dari matriks residual terstandar)
yang tidak bergantung versi `prince`. Hasil biplot sama pada tingkat
konsep.

Langkah:
1. Hitung matriks proporsi $P = N / n$, serta *row mass* $r$ dan
   *column mass* $c$.
2. Matriks residual terstandar:
   $S = D_r^{-1/2}(P - rc^T)D_c^{-1/2}$.
3. SVD: $S = U\Sigma V^T$.
4. Koordinat baris: $D_r^{-1/2}U\Sigma$, koordinat kolom:
   $D_c^{-1/2}V\Sigma$. Inersia (varians) dimensi $k$ adalah
   $\sigma_k^2$.

In [ ]:
def correspondence_analysis(table):
    N = table.values.astype(float)
    n = N.sum()
    P = N / n
    r = P.sum(axis=1)          # row mass
    c = P.sum(axis=0)          # column mass
    S = np.diag(1 / np.sqrt(r)) @ (P - np.outer(r, c)) @ np.diag(1 / np.sqrt(c))
    U, sigma, Vt = np.linalg.svd(S, full_matrices=False)
    rows = np.diag(1 / np.sqrt(r)) @ U * sigma
    cols = np.diag(1 / np.sqrt(c)) @ Vt.T * sigma
    inertia = sigma ** 2
    return rows, cols, inertia

rows, cols, inertia = correspondence_analysis(housetasks)
explained = inertia / inertia.sum()
print('Inersia terjelaskan per dimensi:', explained.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.axhline(0, color='#888888', lw=0.8)
ax.axvline(0, color='#888888', lw=0.8)

ax.scatter(rows[:, 0], rows[:, 1], color='C0', label='Tugas (baris)')
for name, (x, y) in zip(housetasks.index, rows[:, :2]):
    ax.annotate(name, (x, y), color='C0', fontsize=9)

ax.scatter(cols[:, 0], cols[:, 1], color='C1', marker='^', label='Pelaksana (kolom)')
for name, (x, y) in zip(housetasks.columns, cols[:, :2]):
    ax.annotate(name, (x, y), color='C1', fontsize=10, fontweight='bold')

ax.set_xlabel('Dimensi 1 (%.1f%%)' % (100 * explained[0]))
ax.set_ylabel('Dimensi 2 (%.1f%%)' % (100 * explained[1]))
ax.set_title('Correspondence analysis: tugas rumah tangga')
ax.legend()
plt.show()

Pada biplot, tugas dan pelaksana yang berdekatan saling berasosiasi.
Sumbu horizontal membedakan tugas yang terutama dikerjakan **istri**
(misal laundry, memasak makan malam) dari yang dikerjakan **suami**
(misal perbaikan, mengemudi). Sumbu vertikal membedakan tugas yang
dilakukan **bersama** (misal liburan, keuangan) dari yang dilakukan
sendiri.

Sebagai pembanding, pemakaian package `prince` (bila terpasang):

In [ ]:
# Opsional: bandingkan dengan prince (API berbeda antar versi, dibungkus try)
try:
    import prince
    ca = prince.CA(n_components=2).fit(housetasks)
    print('Eigenvalues (inersia) prince:', np.round(ca.eigenvalues_, 4))
    print('Eigenvalues manual          :', np.round(inertia[:2], 4))
except Exception as e:
    print('prince tidak dapat dipakai pada versi ini:', type(e).__name__, e)

### Key Ideas — Correspondence Analysis

- Correspondence analysis memvisualisasikan asosiasi antar kategori
  dari tabel kontingensi.
- Kategori yang berdekatan pada biplot berasosiasi kuat.
- Tidak dipakai untuk reduksi dimensi pada big data, melainkan
  terutama untuk eksplorasi data kategorikal.

# 3. K-Means Clustering

**Clustering** adalah teknik untuk membagi data ke dalam kelompok
berbeda, di mana record dalam tiap kelompok mirip satu sama lain.
Tujuannya mengidentifikasi segmen yang signifikan dan bermakna.
Clustering dapat dipakai langsung, sebagai langkah menuju analisis
lain, atau untuk membangun fitur bagi model.

### Key Terms

- **Cluster** — kelompok record yang mirip.
- **Cluster mean** — vektor rata-rata variabel untuk record dalam
  cluster.
- **K** — jumlah cluster.

**K-means** adalah salah satu metode clustering pertama. Ia membagi
data menjadi K cluster dengan meminimalkan **jumlah kuadrat jarak** tiap
record ke mean cluster-nya, yang disebut **within-cluster sum of
squares** (within-cluster SS, atau inertia):

$$
SS_k = \sum_{i \in \text{cluster } k}\left[(x_i-\bar{x}_k)^2 + (y_i-\bar{y}_k)^2\right]
$$

Algoritma standar (Lloyd): mulai dari K pusat acak, tetapkan tiap
record ke pusat terdekat, hitung ulang pusat sebagai mean cluster,
ulangi sampai tidak ada perubahan.

> K-means tidak menjamin kemungkinan terbaik global. Hasil bergantung
> pada inisialisasi, sehingga algoritma biasanya dijalankan beberapa
> kali dari awal acak berbeda (`n_init`) dan dipilih hasil terbaik.

## Contoh Sederhana

Return saham XOM dan CVX (sejak 2011), dengan 4 cluster.

In [ ]:
df = sp500_px.loc[sp500_px.index >= '2011-01-01', ['XOM', 'CVX']].copy()
kmeans = KMeans(n_clusters=4, n_init=10, random_state=1).fit(df)

df['cluster'] = kmeans.labels_
df.head()

In [ ]:
centers = pd.DataFrame(kmeans.cluster_centers_, columns=['XOM', 'CVX'])
centers

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax = sns.scatterplot(x='XOM', y='CVX', hue='cluster', style='cluster',
                     ax=ax, data=df, palette='tab10')
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
centers.plot.scatter(x='XOM', y='CVX', ax=ax, s=50, color='black')
ax.set_title('K-means (K = 4) pada return XOM dan CVX')
plt.show()

Cluster 1 dan 2 mewakili hari "naik" dan "turun" pasar. Dua cluster lain
mewakili hari dengan pergerakan ekstrem (outlier turun dan naik).
Nomor label cluster bersifat arbitrer dan dapat berbeda antar eksekusi.

## Interpreting the Clusters

Untuk melihat karakter tiap cluster pada banyak variabel, lihat
**cluster means**. Contoh berikut memakai 5 cluster pada return banyak
saham.

In [ ]:
syms = sorted(['AAPL', 'MSFT', 'CSCO', 'INTC', 'CVX', 'XOM', 'SLB', 'COP',
               'JPM', 'WFC', 'USB', 'AXP', 'WMT', 'TGT', 'HD', 'COST'])
top_sp = sp500_px.loc[sp500_px.index >= '2011-01-01', syms]
kmeans = KMeans(n_clusters=5, n_init=10, random_state=1).fit(top_sp)

print('Ukuran cluster:', sorted(Counter(kmeans.labels_).items()))

In [ ]:
centers = pd.DataFrame(kmeans.cluster_centers_, columns=syms)

f, axes = plt.subplots(5, 1, figsize=(5, 8), sharex=True)
for i, ax in enumerate(axes):
    center = centers.loc[i, :]
    maxPC = 1.01 * np.max(np.max(np.abs(center)))
    colors = ['C0' if l > 0 else 'C1' for l in center]
    ax.axhline(color='#888888')
    center.plot.bar(ax=ax, color=colors)
    ax.set_ylabel(f'Cluster {i + 1}')
    ax.set_ylim(-maxPC, maxPC)
plt.tight_layout()
plt.show()

Cluster yang terbentuk menggambarkan pola pasar yang berbeda:
sebagian cluster mewakili hari ketika saham energi naik dan lainnya
turun, sebagian lagi hari ketika seluruh pasar turun/naik kuat, dan
sebagian cluster kecil mewakili hari ekstrem. Cluster yang sangat kecil
dengan perubahan besar menunjukkan hari-hari dengan peristiwa khusus.

## Selecting the Number of Clusters

K-means mengharuskan kita menentukan jumlah K. Pendekatan umum adalah
**elbow method**: pilih K ketika penambahan cluster tidak lagi banyak
menurunkan persen varians yang dijelaskan. Persentase varians yang
dijelaskan = rasio within-cluster SS terhadap total SS. Dengan K
bertambah, inertia turun terus; "siku" ada di titik ketika penurunan
berkurang tajam. Pada praktiknya siku tidak selalu jelas, dan pilihan K
sering ditentukan kombinasi elbow dan kebutuhan bisnis.

In [ ]:
inertia = []
for n_clusters in range(2, 14):
    km = KMeans(n_clusters=n_clusters, n_init=10, random_state=0).fit(top_sp)
    inertia.append(km.inertia_ / n_clusters)

inertias = pd.DataFrame({'n_clusters': range(2, 14), 'inertia': inertia})
ax = inertias.plot(x='n_clusters', y='inertia', marker='o', figsize=(5, 4), legend=False)
plt.xlabel('Number of clusters (k)')
plt.ylabel('Average Within-Cluster Squared Distances')
plt.ylim((0, 1.1 * inertias.inertia.max()))
plt.title('Elbow method')
plt.show()

Pada contoh ini tidak ada siku yang jelas. Hal ini umum pada data yang
tidak punya struktur cluster yang tegas. Menurut buku, K-means tetap
berguna untuk **memotong data** menjadi segmen praktis walau tidak
ada klaster alami.

> **Cluster Analysis versus PCA.** Keduanya meringkas data: PCA
> mengurangi jumlah variabel, clustering mengurangi jumlah record
> (mengelompokkan menjadi segmen). Kita juga bisa memakai
> cluster means atau skor PCA sebagai ringkasan.

### Key Ideas — K-Means

- Jumlah cluster K ditentukan pengguna.
- Algoritma berkembang secara iteratif dengan menetapkan record ke
  cluster terdekat sampai tidak ada perubahan.
- Praktik K-means dalam data science umumnya memotong data menjadi
  segmen berguna, bukan menemukan cluster alami.
- Elbow method membantu memilih K tetapi tidak selalu jelas.

# 4. Hierarchical Clustering

**Hierarchical clustering** adalah alternatif untuk K-means yang dapat
menghasilkan cluster berbeda dan memungkinkan visualisasi hubungan
hierarkis antar cluster. Keunggulannya:

- Tidak perlu menentukan jumlah cluster sejak awal.
- Dapat menghasilkan **dendrogram** yang menunjukkan struktur hierarki.
- Dapat digunakan pada data dengan banyak fitur bila matriks jarak
  tersedia (termasuk data campuran lewat Gower's distance).
- Bersifat deterministik (tidak bergantung inisialisasi acak).

Kekurangannya: tidak skalabel ke dataset besar karena membutuhkan
matriks jarak antar record (kompleksitas besar).

### Key Terms

- **Dendrogram** — representasi visual record dan hierarki cluster
  yang dikandungnya.
- **Distance** — ukuran seberapa dekat satu record dengan record lain.
- **Dissimilarity** — ukuran seberapa dekat satu cluster dengan
  cluster lain.

## A Simple Example

Hierarchical clustering bekerja pada dataset dengan n record dan p
variabel, berdasarkan dua blok bangunan dasar:

- Sebuah **metrik jarak** $d_{i,j}$ untuk mengukur jarak antara dua
  record i dan j.
- Sebuah **metrik dissimilarity** $D_{A,B}$ untuk mengukur perbedaan
  antar dua cluster A dan B berdasarkan jarak antar anggotanya.

Untuk data numerik, metrik jarak biasanya Euclidean. Untuk dissimilarity
ada beberapa pilihan (linkage).

Algoritma aglomeratif:
1. Mulai dengan setiap record sebagai cluster sendiri.
2. Hitung dissimilarity antar semua pasangan cluster.
3. Gabungkan dua cluster A dan B yang paling kecil dissimilarity-nya.
4. Ulangi langkah 2-3 sampai semua menjadi satu cluster.

Berikut clustering hierarkis pada return **perusahaan** (transposed:
tiap baris adalah satu saham).

In [ ]:
syms1 = ['AAPL', 'AMZN', 'AXP', 'COP', 'COST', 'CSCO', 'CVX', 'GOOGL', 'HD',
         'INTC', 'JPM', 'MSFT', 'SLB', 'TGT', 'USB', 'WFC', 'WMT', 'XOM']
df = sp500_px.loc[sp500_px.index >= '2011-01-01', syms1].transpose()

Z = linkage(df, method='complete')

fig, ax = plt.subplots(figsize=(6, 5))
dendrogram(Z, labels=df.index, ax=ax, color_threshold=0)
plt.xticks(rotation=90)
ax.set_ylabel('distance')
ax.set_title('Dendrogram (complete linkage)')
plt.show()

**Dendrogram** menunjukkan hierarki: cabang yang bergabung pada
ketinggian rendah lebih mirip. Misalnya saham perusahaan minyak (COP,
CVX, XOM, SLB) bergabung dahulu, begitu juga saham keuangan (JPM, WFC,
USB, AXP). Panjang cabang menunjukkan jarak antar cluster; "memotong"
dendrogram pada ketinggian tertentu menghasilkan jumlah cluster
tertentu.

Memotong tree menjadi 4 cluster:

In [ ]:
memb = fcluster(Z, 4, criterion='maxclust')
memb = pd.Series(memb, index=df.index)
for key, item in memb.groupby(memb):
    print(f"{key} : {', '.join(item.index)}")

## The Agglomerative Algorithm dan Ukuran Dissimilarity

Empat ukuran dissimilarity (linkage) yang umum:

| Linkage | Definisi $D_{A,B}$ | Sifat |
|---|---|---|
| **Complete** | maksimum jarak antara anggota A dan B | cenderung membentuk cluster kompak berukuran seimbang |
| **Single** | minimum jarak antara anggota A dan B | cenderung membentuk cluster memanjang ("chaining"), sensitif noise |
| **Average** | rata-rata semua jarak pasangan A-B | kompromi antara single dan complete |
| **Minimum variance (Ward)** | menggabungkan cluster yang paling sedikit menambah total within-cluster SS | mirip K-means, cluster kompak |

Buku menjelaskan bahwa complete menghasilkan cluster kompak, single
sering menghasilkan cluster "rantai" yang tidak seimbang, sedangkan
average dan Ward berada di antaranya. Ward paling mirip tujuan
K-means.

In [ ]:
# Bandingkan beberapa linkage pada data saham yang sama
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, method in zip(axes, ['single', 'complete', 'average', 'ward']):
    Zm = linkage(df, method=method)
    dendrogram(Zm, labels=df.index, ax=ax, color_threshold=0)
    ax.set_title(method)
    ax.tick_params(axis='x', rotation=90)
plt.tight_layout()
plt.show()

### Key Ideas — Hierarchical Clustering

- Hierarchical clustering dimulai dengan tiap record sebagai cluster
  sendiri dan menggabungkan secara iteratif cluster yang terdekat.
- Hasilnya divisualisasikan dengan dendrogram.
- Beberapa ukuran dissimilarity (complete, single, average, Ward)
  menghasilkan hasil berbeda.
- Tidak perlu menentukan K di awal, tetapi tidak skalabel ke data besar.

# 5. Model-Based Clustering

Metode clustering seperti K-means dan hierarchical bersifat heuristik,
berdasarkan algoritma, bukan model probabilistik. **Model-based
clustering** didasarkan pada teori statistik formal dan memberi cara
lebih ketat menentukan sifat dan jumlah cluster.

### Key Terms

- **Mixture model** — asumsi bahwa data berasal dari campuran dua atau
  lebih distribusi.
- **Multivariate normal distribution** — generalisasi distribusi
  normal ke lebih dari satu variabel.
- **Covariance matrix** — matriks yang menjelaskan bagaimana dua atau
  lebih variabel berkorelasi.
- **BIC (Bayesian Information Criteria)** — ukuran kualitas model yang
  menghukum jumlah parameter; semakin kecil (pada sklearn) semakin baik.

## Multivariate Normal Distribution

Distribusi normal multivariat ditentukan oleh **vektor mean** $\mu$
dan **matriks kovarians** $\Sigma$. Kovarians menyatakan bagaimana
variabel berkorelasi (untuk variabel yang distandardisasi, kovarians
adalah korelasi). Kontur distribusi ini berbentuk elips; orientasi dan
bentuk elips ditentukan oleh $\Sigma$.

## Mixtures of Normals

Ide kunci: data berasal dari **campuran beberapa distribusi normal
multivariat**, masing-masing dengan mean dan kovarians sendiri.
Algoritma mencari kombinasi parameter (mean, kovarians, proporsi
campuran) yang paling mungkin menghasilkan data (maximum likelihood,
dengan algoritma **EM / expectation-maximization**) lalu setiap record
ditetapkan ke komponen dengan probabilitas posterior tertinggi.

Contoh: dua komponen pada return XOM dan CVX.

In [ ]:
df = sp500_px.loc[sp500_px.index >= '2011-01-01', ['XOM', 'CVX']]
mclust = GaussianMixture(n_components=2, random_state=1).fit(df)

fig, ax = plt.subplots(figsize=(5, 5))
colors = [f'C{c}' for c in mclust.predict(df)]
df.plot.scatter(x='XOM', y='CVX', c=colors, alpha=0.5, ax=ax)
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_title('Gaussian mixture dengan 2 komponen')
plt.show()

In [ ]:
print('Mean')
print(mclust.means_)
print('Covariances')
print(mclust.covariances_)

Satu cluster berpusat di dekat nol dengan varians kecil (hari-hari
tenang), cluster lain juga berpusat di nol tetapi dengan varians besar
(hari volatil). Ini contoh model-based clustering yang membedakan
cluster berdasarkan **sebaran (kovarians)**, bukan hanya lokasi, hal
yang tidak dapat ditangkap K-means.

## Selecting the Number of Clusters

Jumlah komponen dipilih dengan **BIC**. Hanya model-based clustering
yang memberi dasar statistik formal untuk memilih jumlah cluster.
Pada scikit-learn, BIC yang lebih **kecil** lebih baik (tanda berbeda
dari `mclust` di R yang memaksimalkan). Selain jumlah komponen, kita
juga memilih tipe kovarians: `full`, `tied`, `diag`, `spherical`.

In [ ]:
results = []
covariance_types = ['full', 'tied', 'diag', 'spherical']
for n_components in range(1, 9):
    for covariance_type in covariance_types:
        mc = GaussianMixture(n_components=n_components, warm_start=True,
                             covariance_type=covariance_type, random_state=1)
        mc.fit(df)
        results.append({
            'bic': mc.bic(df),
            'n_components': n_components,
            'covariance_type': covariance_type,
        })

results = pd.DataFrame(results)

colors = ['C0', 'C1', 'C2', 'C3']
styles = ['C0-', 'C1:', 'C0-.', 'C1--']

fig, ax = plt.subplots(figsize=(5, 4))
for i, covariance_type in enumerate(covariance_types):
    subset = results.loc[results.covariance_type == covariance_type, :]
    subset.plot(x='n_components', y='bic', ax=ax, label=covariance_type,
                kind='line', style=styles[i])
ax.set_ylabel('BIC (lebih kecil lebih baik)')
ax.set_title('BIC untuk jumlah komponen dan tipe kovarians')
plt.show()

best = results.loc[results.bic.idxmin()]
print('Model terbaik menurut BIC:', best.to_dict())

Plot BIC mirip dengan plot elbow K-means, kecuali yang digambar BIC,
bukan persen varians. Perbedaan penting: plot ini menunjukkan satu garis
untuk setiap tipe kovarians. Model terbaik adalah kombinasi yang
BIC-nya paling kecil (pada sklearn).

> **Mixture Models Asumsi dan Batas.** Model-based clustering
> mengasumsikan setiap cluster berdistribusi normal multivariat. Jika
> data jauh dari normal, atau banyak variabel, hasilnya bisa kurang
> andal. Metode ini juga lebih berat secara komputasi dibanding
> K-means.

### Key Ideas — Model-Based Clustering

- Cluster diasumsikan berasal dari proses probabilistik berbeda.
- Pemakaian distribusi normal multivariat adalah yang paling umum.
- Jumlah cluster dipilih memakai BIC.
- Cocok bila ingin memisahkan cluster yang berbeda dari sisi sebaran,
  bukan hanya lokasi pusatnya.

# 6. Scaling and Categorical Variables

Teknik unsupervised learning umumnya mengharuskan data diskalakan
(*scaled*). Berbeda dengan teknik di bab sebelumnya (regression,
tree) yang tidak sensitif terhadap skala, K-means, PCA, dan
clustering lain sangat dipengaruhi skala variabel.

### Key Terms

- **Scaling** — memperbesar atau memperkecil data agar berada pada
  skala yang sama.
- **Normalization** — salah satu metode scaling: kurangi mean lalu
  bagi dengan standard deviation. Sinonim: standardization.
- **Gower's distance** — algoritma penskalaan yang menghasilkan
  jarak 0-1 untuk data numerik dan kategorikal campuran.

## Scaling the Variables

Variabel dengan skala sangat berbeda dapat mendominasi. Contoh:
K-means pada data pinjaman *default* dengan variabel `loan_amnt`,
`annual_inc`, `revol_bal`, `open_acc`, `dti`, `revol_util`. Perhatikan
perbedaan hasil **tanpa** dan **dengan** standardisasi.

In [ ]:
defaults = loan_data.loc[loan_data['outcome'] == 'default', ]

columns = ['loan_amnt', 'annual_inc', 'revol_bal', 'open_acc',
           'dti', 'revol_util']

df = defaults[columns]
kmeans = KMeans(n_clusters=4, n_init=10, random_state=1).fit(df)
counts = Counter(kmeans.labels_)

centers = pd.DataFrame(kmeans.cluster_centers_, columns=columns)
centers['size'] = [counts[i] for i in range(4)]
print('Tanpa scaling:')
centers.round(2)

Tanpa scaling, cluster hampir sepenuhnya ditentukan oleh `annual_inc`
dan `revol_bal` (variabel berskala besar). Satu cluster hanya berisi
sedikit record dengan pendapatan sangat tinggi, dan cluster lain
terutama dibedakan oleh pendapatan, sementara variabel seperti `dti`
dan `open_acc` hampir tidak berperan.

Sekarang dengan standardisasi (`StandardScaler`), dengan pusat
cluster ditransformasi balik ke skala asli agar bisa diinterpretasi.

In [ ]:
scaler = preprocessing.StandardScaler()
df0 = scaler.fit_transform(df * 1.0)

kmeans = KMeans(n_clusters=4, n_init=10, random_state=1).fit(df0)
counts = Counter(kmeans.labels_)

centers = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_),
                       columns=columns)
centers['size'] = [counts[i] for i in range(4)]
print('Dengan scaling:')
centers.round(2)

Setelah scaling, ukuran cluster jauh lebih seimbang dan cluster
dibedakan oleh banyak variabel, bukan hanya pendapatan dan saldo.

## Dominant Variables

Meskipun variabel sudah dinormalisasi, satu atau dua variabel bisa
tetap mendominasi komponen. Contoh pada PCA: saham **GOOGL** dan
**AMZN** bergerak sangat berbeda dari saham lain (pergerakan idiosinkratik
besar), sehingga mendominasi komponen. Perhatikan screeplot dan loadings
setelah GOOGL dan AMZN dimasukkan.

In [ ]:
syms = ['GOOGL', 'AMZN', 'AAPL', 'MSFT', 'CSCO', 'INTC', 'CVX', 'XOM',
        'SLB', 'COP', 'JPM', 'WFC', 'USB', 'AXP', 'WMT', 'TGT', 'HD', 'COST']
top_sp1 = sp500_px.loc[sp500_px.index >= '2005-01-01', syms]

sp_pca1 = PCA()
sp_pca1.fit(top_sp1)

explained_variance = pd.DataFrame(sp_pca1.explained_variance_)
ax = explained_variance.head(10).plot.bar(legend=False, figsize=(5, 4))
ax.set_xlabel('Component')
ax.set_ylabel('Explained variance')
ax.set_title('Screeplot dengan GOOGL dan AMZN')
plt.show()

In [ ]:
loadings = pd.DataFrame(sp_pca1.components_[0:2, :], columns=top_sp1.columns,
                        index=['PC1', 'PC2'])
loadings.T.round(3)

Dua komponen pertama hampir didominasi oleh GOOGL dan AMZN. Pada
komponen pertama GOOGL mendapat bobot besar, pada komponen kedua AMZN.
Artinya variasi pada dua saham ini mendominasi struktur data, bukan
pola umum pasar. Dalam kasus seperti ini ada dua pilihan: terima bahwa
variabel tersebut memang penting, atau keluarkan atau kurangi
bobotnya. Pada PCA/clustering, pembahasan variabel dominan sering
menyingkap **outlier** yang perlu dipelajari tersendiri.

## Categorical Data and Gower's Distance

Pada data campuran numerik dan kategorikal, jarak Euclidean tidak
dapat dipakai langsung. Salah satu solusi: ubah kategorikal menjadi
dummy (one hot encoding) lalu pakai K-means, namun hasilnya sering
bermasalah (lihat bagian berikut). Alternatif adalah **Gower's
distance**.

Gower's distance menskalakan semua jarak variabel ke 0-1 lalu merata-
ratakannya:

- **Variabel numerik atau ordinal**: jarak absolut antara dua record
  dibagi dengan **rentang (range)** variabel tersebut.
- **Variabel kategorikal**: jarak 1 jika kategori berbeda, 0 jika
  sama.
- Jarak Gower adalah **rata-rata** jarak semua variabel.

Contoh pada lima pinjaman pertama dengan variabel `dti`,
`payment_inc_ratio`, `home_`, dan `purpose_`.

In [ ]:
x = loan_data.loc[0:4, ['dti', 'payment_inc_ratio', 'home_', 'purpose_']]
x

Pada buku, jarak dihitung dengan fungsi `daisy(x, metric='gower')`
dari package `cluster` di R. Saat buku ditulis, Gower's distance tidak
tersedia di package Python populer, sehingga di sini dibuat
**implementasi manual** yang menghasilkan nilai sama.

In [ ]:
def gower_distance(data):
    # Gower's distance antar seluruh baris DataFrame (numerik + kategorikal)
    n = len(data)
    total = np.zeros((n, n))
    for col in data.columns:
        s = data[col]
        if pd.api.types.is_numeric_dtype(s):
            v = s.astype(float).values
            rng = v.max() - v.min()
            d = np.abs(v[:, None] - v[None, :]) / rng if rng > 0 else np.zeros((n, n))
        else:
            v = pd.factorize(s.astype(str))[0]     # kode integer per kategori
            d = (v[:, None] != v[None, :]).astype(float)
        total += d
    return total / data.shape[1]

D = gower_distance(x)
pd.DataFrame(D, index=x.index, columns=x.index).round(7)

Nilai ini cocok dengan keluaran `daisy` pada buku untuk lima record
tersebut, misalnya jarak (2,1) = 0,6220479, (3,2) = 0,8143398, dan
(5,3) = 0,3091088.

Semua jarak berada antara 0 dan 1. Pasangan dengan jarak terbesar
adalah record 2 dan 3: tidak ada nilai `home_` dan `purpose_` yang sama
dan `dti` serta `payment_inc_ratio` sangat berbeda. Record 3 dan 5
memiliki jarak terkecil karena berbagi nilai `home_` dan `purpose_`.

Matriks jarak Gower dapat diteruskan ke hierarchical clustering. Pada
contoh berikut dipakai 250 sampel pinjaman *default*.

In [ ]:
df_g = defaults.sample(250, random_state=1)[['dti', 'payment_inc_ratio',
                                              'home_', 'purpose_']].reset_index(drop=True)
D = gower_distance(df_g)

Zg = linkage(squareform(D, checks=False), method='complete')

fig, ax = plt.subplots(figsize=(7, 4))
dendrogram(Zg, no_labels=True, ax=ax, color_threshold=0)
ax.set_title("Dendrogram hierarchical clustering dengan Gower's distance")
plt.show()

cut = fcluster(Zg, 5, criterion='maxclust')
df_g[cut == cut[0]].head()

Dendrogram menunjukkan bahwa kelompok-kelompok besar dapat dipisahkan,
dan record dalam satu cluster berbagi nilai kategori serta numerik yang
mirip.

## Problems with Clustering Mixed Data

K-means dan PCA paling cocok untuk data numerik kontinu. Untuk data
campuran, kita bisa mengubah faktor menjadi dummy variable dan
menormalisasi semuanya, tetapi hasilnya bermasalah:

- Dummy biner (0/1) setelah standardisasi memberikan jarak yang tidak
  alami, dan variabel kategorikal dengan banyak level menghasilkan
  banyak dummy sehingga kategorikal **mendominasi** jarak.
- Cluster cenderung terbentuk oleh dummy variable (misal satu kategori
  tertentu) dibanding variabel numerik.

Contoh: K-means pada `dti`, `payment_inc_ratio`, `home_`, `pub_rec_zero`
setelah one hot encoding dan standardisasi.

In [ ]:
columns = ['dti', 'payment_inc_ratio', 'home_', 'pub_rec_zero']
if 'pub_rec_zero' not in defaults.columns:
    defaults = defaults.assign(pub_rec_zero=(defaults['pub_rec'] == 0).astype(int))
df = pd.get_dummies(defaults[columns].astype({'home_': str}), dtype=float)

scaler = preprocessing.StandardScaler()
df0 = scaler.fit_transform(df * 1.0)
kmeans = KMeans(n_clusters=4, n_init=10, random_state=1).fit(df0)

centers = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_),
                       columns=df.columns)
centers.round(2)

Empat cluster hasil K-means terutama ditentukan oleh variabel
kategorikal: satu cluster hampir seluruhnya berisi pemilik rumah
tertentu (`home_OWN`, `home_MORTGAGE`, `home_RENT`), sedangkan variabel
numerik `dti` dan `payment_inc_ratio` hampir tidak membedakan
cluster. Pada data campuran, pendekatan dummy plus K-means kurang baik
dan **hierarchical clustering dengan Gower's distance** lebih tepat.

### Key Ideas — Scaling dan Categorical Variables

- Variabel harus diskalakan agar variabel berskala besar tidak
  mendominasi hasil.
- Variabel dominan dapat memengaruhi PCA dan clustering.
- Untuk data campuran, Gower's distance menghasilkan jarak 0-1 yang
  menggabungkan numerik dan kategorikal.
- K-means pada dummy variable cenderung didominasi variabel kategorikal.

# Key Ideas — Chapter 7

### 1. Unsupervised Learning
Mengekstrak struktur dari data tanpa variabel respons: reduksi
dimensi dan pengelompokan (clustering).

### 2. PCA
Mengubah variabel numerik yang berkorelasi menjadi sedikit komponen
tidak berkorelasi yang menjelaskan sebagian besar varians. Screeplot
dan loadings dipakai untuk interpretasi.

### 3. Correspondence Analysis
Biplot asosiasi antar kategori dari tabel kontingensi.

### 4. K-Means
Membagi data ke K cluster dengan meminimalkan within-cluster SS. K
dipilih dengan elbow method.

### 5. Hierarchical Clustering
Penggabungan bertahap menghasilkan dendrogram; hasil dipengaruhi
pilihan linkage (complete, single, average, Ward).

### 6. Model-Based Clustering
Mixture of multivariate normal; jumlah cluster dan tipe kovarians
dipilih dengan BIC.

### 7. Scaling dan Gower's Distance
Standardisasi diperlukan; untuk data campuran dipakai Gower's distance
bersama hierarchical clustering.

## Kesimpulan

Chapter 7 membahas metode unsupervised learning yang menemukan struktur
dalam data tanpa label. **PCA** dan **correspondence analysis**
mereduksi atau memvisualisasikan struktur variabel numerik dan
kategorikal. **K-means**, **hierarchical clustering**, dan
**model-based clustering** mengelompokkan record, masing-masing dengan
kelebihan: K-means sederhana dan cepat, hierarchical memberi
dendrogram tanpa menetapkan K di awal, model-based memberi dasar
statistik formal untuk memilih jumlah cluster.

Hal penting dalam praktik adalah **scaling**: variabel harus
dinormalisasi agar tidak ada variabel yang mendominasi, dan data
campuran numerik-kategorikal sebaiknya ditangani dengan Gower's
distance. Unsupervised learning sangat berguna untuk eksplorasi dan
untuk membuat fitur bagi model supervised.